# Multi-Agent News Brief

Find news, filter it with an editor, write a script, and generate a voice briefing with OpenAI or Gemini.

## Workflow overview

This workflow searches for recent news, uses two agents to refine it into a script, then creates an audio briefing. Run the next cell to render the diagram.

In [15]:
%%html
<script src="https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js"></script>

<div class="mermaid">
flowchart TD
    A[Choose a news topic] --> B[DuckDuckGo news search]
    B --> C[News Editor agent: filter and refine]
    C --> D[News Script Writer agent]
    D --> E{Selected provider}
    E -->|OpenAI| F[OpenAI TTS: MP3]
    E -->|Gemini| G[Gemini TTS: WAV]
    F --> H[Play and download audio]
    G --> H
</div>

<script>
  mermaid.initialize({ startOnLoad: false, theme: 'neutral' });
  mermaid.run({ querySelector: '.mermaid' });
</script>

## 1. Install dependencies

In [16]:
%pip install -q openai-agents ddgs google-genai

## 2. Select an agent provider

Set `PROVIDER` to `"openai"` or `"gemini"`. The same provider also generates the final voice briefing.

In [17]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "gemini"  # Change to "openai" to use OpenAI for agents and TTS.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-2.5-flash"

if PROVIDER == "openai":
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets for Gemini agents.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Find recent news

In [18]:
from ddgs import DDGS


def search_news(query: str) -> str:
    """Find recent news and return each item's title, summary, date, and URL."""
    results = DDGS(timeout=15).news(query, timelimit="d", max_results=10)
    if not results:
        raise RuntimeError("No recent news results were found. Try a different topic.")

    for index, item in enumerate(results, start=1):
        print(f"{index}. {item.get('title', 'Untitled')}")
        print(item.get('url', 'No URL available.'))

    return "\n\n".join(
        f"Title: {item.get('title', 'Untitled')}\nDate: {item.get('date', 'Unknown')}\n"
        f"Summary: {item.get('body', 'No summary available.')}\nURL: {item.get('url', '')}"
        for item in results
    )


topic = "artificial intelligence business"
news_items = search_news(topic)

1. Small Businesses Using AI Are Hiring More, Not Less. New Study Finds a 10% Boost...
https://www.ibtimes.com/small-businesses-using-ai-are-hiring-more-not-less-new-study-finds-10-boost-smallest-firms-3807341
2. New iOS 27 Code Hints At Sponsored Ads In Visual Intelligence
https://www.macobserver.com/news/new-ios-27-code-hints-at-sponsored-ads-in-visual-intelligence/
3. N.C. A&T lands inaugural Department of Defense $10M grant to advance critical AI...
https://www.bizjournals.com/triad/news/2026/09/11/ncat-defense-department-grant-hbcu-accelerate-daq.html
4. UNT Gets $20 Million Gift To Launch AI And Advanced Analytics College
https://www.yahoo.com/news/science/articles/unt-gets-20-million-gift-150013803.html
5. AI Is Boosting the U.K. Economy
https://www.wsj.com/livecoverage/stock-market-cpi-inflation-09-11-2026/card/ai-is-boosting-the-u-k-economy-5eybjlSlW8qgAcALkCij
6. OpenAI and Anthropic top our list of private companies, but their reign will be...
https://www.bizjournals.com/san

## 4. Editor Agent: filter and refine

In [19]:
from agents import Agent, Runner

editor_agent = Agent(
    name="News Editor",
    instructions="Select the three most relevant, credible, non-duplicative items. Exclude ads, clickbait, speculation, and weak evidence. Preserve each selected item's key facts, date, and URL.",
    model=model,
)

editor_result = await Runner.run(editor_agent, f"Topic: {topic}\n\nNews items:\n{news_items}")
edited_news = editor_result.final_output
print(edited_news)

- OpenAI and Anthropic top our list of private companies, but their reign will be... — Key facts: OpenAI and Anthropic rank as the fastest-growing private companies on a recent list; software developers dominate the ranking. Date: 2026-09-11T12:02:58+00:00. URL: https://www.bizjournals.com/sanfrancisco/news/2026/09/11/private-companies-list-openai-anthropic-ipo.html

- AI Is Boosting the U.K. Economy — Key facts: July GDP unexpectedly rose with notable gains in computer programming and related activity, signaling measurable economic impact from AI-driven sectors. Date: 2026-09-11T12:07:58+00:00. URL: https://www.wsj.com/livecoverage/stock-market-cpi-inflation-09-11-2026/card/ai-is-boosting-the-u-k-economy-5eybjlSlW8qgAcALkCij

- N.C. A&T lands inaugural Department of Defense $10M grant to advance critical AI... — Key facts: N.C. A&T received a five-year, $10 million DoD grant to fund research in artificial intelligence and quantum computing. Date: 2026-09-11T12:04:58+00:00. URL: https:

## 5. Writer Agent: create the script

In [20]:
writer_agent = Agent(
    name="News Script Writer",
    instructions="Write a neutral 45- to 60-second spoken news script using only the editor's selected items. Do not add unsupported facts. End by naming the source publications without reading URLs aloud.",
    model=model,
)

writer_result = await Runner.run(writer_agent, f"Create a spoken news script from this edited brief:\n\n{edited_news}")
news_script = writer_result.final_output
print(news_script)

OpenAI and Anthropic are ranked as the fastest-growing private companies on a recent list, with software developers dominating the rankings. In the U.K., July GDP unexpectedly rose, with notable gains in computer programming and related activity signaling a measurable economic impact from AI-driven sectors. And in North Carolina, N.C. A&T has received a five-year, $10 million Department of Defense grant to fund research in artificial intelligence and quantum computing. Stories from BizJournals and The Wall Street Journal.


## 6. Generate the voice briefing

This creates AI-generated speech with the selected provider. Disclose that the voice is AI-generated when sharing it.

In [21]:
from IPython.display import Audio, display

if PROVIDER == "openai":
    tts_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    speech = await tts_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="coral",
        input=news_script,
        instructions="Speak clearly in a calm, neutral broadcast-news style.",
    )
    audio_path = "news_brief.mp3"
    speech.write_to_file(audio_path)
else:
    import base64
    import wave

    from google import genai

    def save_wav(filename, pcm, channels=1, rate=24000, sample_width=2):
        with wave.open(filename, "wb") as wav_file:
            wav_file.setnchannels(channels)
            wav_file.setsampwidth(sample_width)
            wav_file.setframerate(rate)
            wav_file.writeframes(pcm)

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    response = gemini_client.interactions.create(
        model="gemini-3.1-flash-tts-preview",
        input=(
            "Read this in a calm, neutral broadcast-news style:\n\n"
            f"{news_script}"
        ),
        response_format={"type": "audio"},
        generation_config={"speech_config": [{"voice": "Kore"}]},
    )
    audio_path = "news_brief.wav"
    save_wav(audio_path, base64.b64decode(response.output_audio.data))

display(Audio(audio_path))


## 7. Download the audio file

Run this cell to save the generated MP3 (OpenAI) or WAV (Gemini) file to your computer.

In [22]:
from google.colab import files

files.download(audio_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>